# Scrapers for MyDramaList

**Table of Contents**

1. [Research Question](#1)
2. [Scraping Aggregation Page for Topic Movies](#2)
3. [Scraping Page for a Particular Movie](#3)


<a id="1"></a>

## Research Question

<a id="pattern"></a>
My focus is: **which release years appear most often among the top-ranked movies?** 

My motivation for this research question is recency bias. I will collect each movie's title, rank, and release year.

<a id="2"></a>
## Aggregation Page

In [89]:
import math
import csv

from bs4 import BeautifulSoup
from seleniumbase import Driver

url = "https://mydramalist.com/movies/top"

`pase_box()` will extract the values of the box. Specifically, I will extract the movie title, movie ranking, and other movie details including movie type and year.

details_element also gives the movie country, but since I am only interested in the release year, I will ignore that part.

In [ ]:
def parse_box(card):
    # Extract a movie's title, ranking, and release year.
    title_element = card.find("h6", class_="text-primary title")
    ranking_element = card.find("div", class_="ranking pull-right")
    details_element = card.find("span", class_="text-muted")

    return (
        title_element.get_text(strip=True),
        ranking_element.get_text(strip=True),
        details_element.get_text(strip=True).split(" - ")[1] # Extract the release year from the details string
    )


In [87]:
movies = []

with Driver(page_load_strategy="eager") as driver:
    driver.open(url)

    # 1. Extract total results count (e.g., "105 results" -> 105)
    total_text = driver.get_text("p.pull-right") # This will be a string value
    total_results = int(total_text.split()[0])

    # 2. Calculate total pages (20 items per page)
    total_pages = math.ceil(total_results / 20)
    print(f"Total results: {total_results} | Total pages: {total_pages}")

    # 4. Loop through each page URL
    for page in range(1, total_pages + 1):
        driver.open(f"{url}?page={page}")
        driver.sleep(1.0)

        # 5. Extract items on the current page
        rendered_html = driver.get_page_source()
        soup = BeautifulSoup(rendered_html, "html.parser")

        big_box = soup.find("div", class_="m-t nav-active-border b-primary")
        cards = big_box.find_all( "div",class_="box")

        for card in cards:
            movie = parse_box(card)
            movies.append(movie)
            print(movie)

  
        print(f"Page {page}: Scraped {len(cards)} dramas")

        

Total results: 5000 | Total pages: 250
('Better Days', '#1', '2019')
('Miracle in Cell No. 7', '#2', '2013')
('Hope', '#3', '2013')
('How to Make Millions before Grandma Dies', '#4', '2024')
('Monster', '#5', '2023')
('Train to Busan', '#6', '2016')
('Drawing Closer', '#7', '2024')
('A Taxi Driver', '#8', '2017')
('Parasite', '#9', '2019')
('Lighting Up the Stars', '#10', '2022')
('Silenced', '#11', '2011')
('The Paradise of Thorns', '#12', '2024')
('The Man from Nowhere', '#13', '2010')
('Rurouni Kenshin: The Legend Ends', '#14', '2014')
('Sunny', '#15', '2011')
('Your Eyes Tell', '#16', '2020')
('Even if This Love Disappears from the World Tonight', '#17', '2022')
('Our Times', '#18', '2015')
('Midnight Runners', '#19', '2017')
('Ode to My Father', '#20', '2014')
Page 1: Scraped 20 dramas
('Be with You', '#21', '2018')
('Along with the Gods 2: The Last 49 Days', '#22', '2018')
('Rurouni Kenshin: Kyoto Inferno', '#23', '2014')
('My Annoying Brother', '#24', '2016')
('Seven Samurai', '

Finally, I need to save the output into a csv file.

In [88]:
with open("top_movies.csv","w") as file:
    writer = csv.writer(file)
    writer.writerow(["title", "rank", "release_year"])
    writer.writerows(movies)

print(
    f"Saved {len(movies)} movies to top_movies.csv"
)

Saved 5000 movies to top_movies.csv


<a id="3"></a>
## Scraping Page for a Particular Movie

The movie I chose is Ip Man (I loved watching it as a kid). Therefore, I will define the url.

In [90]:
url2 = "https://mydramalist.com/516-ip-man"

In [111]:
with Driver(page_load_strategy="eager") as driver:
    driver.open(url2)
    driver.wait_for_element("body")
    ipman_html = driver.get_page_source()

I then define the function to parse the data.

In [ ]:
def parse_ipman(html_content):
    #Extracts the Details section from the Ip Man page.

    if not html_content:
        return []

    soup = BeautifulSoup(html_content, "html.parser")
    details = soup.find("ul", class_= "list m-b-0") 
    details_list = details.find_all("li") #if I include class here, content rating field will not show. Therefore, I will onlyinclude "li"


    details_data = []
    for info in details_list: # interate through each field in the detail section
        data = info.get_text(" ", strip=True) # avoid no space between the field name and its value. For example, "Content Rating: 13+" will become "Content Rating:13+" if I don't include " " in get_text()
        details_data.append(data)

    return details_data


In [116]:
ipman_details = parse_ipman(ipman_html)
print(ipman_details)

['Title: Ip Man', 'Type: Movie', 'Format: Feature Film', 'Country: Hong Kong', 'Release Date: Dec 12, 2008', 'Duration: 1 hr. 46 min.', 'Content Rating: Not Yet Rated']


I recoreded the scraped results to a csv.

In [117]:
with open("ip_man_details.csv","w") as f:
    writer = csv.writer(f)
    writer.writerow(["info", "value"])
    for detail in ipman_details:
            info, value = detail.split(":", 1)
            writer.writerow([info.strip(), value.strip()])

print(
    f"Saved {len(ipman_details)} details to ip_man_details.csv"
)

Saved 7 details to ip_man_details.csv
